Storage proof periods
---------------------

This document helps when choosing [proof timing][1] parameters for the
[marketplace smart contract][2]. Based on a choice of period length, it
calculates the odds that a storage provider needs to calculate more proofs than
it is capable of. It also calculates the downtime parameters.

[1]: https://github.com/promethei-project/promethei-research-old/blob/master/design/storage-proof-timing.md
[2]: https://github.com/promethei-project/promethei-contracts/blob/0c56e28dd5db75e50766033af36c016f9e663e60/configuration/configuration.js#L12


Timing assumptions:

In [9]:
from datetime import timedelta

period_length = timedelta(minutes=5)
period_timeout = timedelta(minutes=1)
proving_time = timedelta(seconds=12)
proof_frequency = timedelta(days=1)

print(f'Proof period length: {period_length}')
print(f'Proof validator timeout: {period_timeout}')
print(f'Calculating a proof takes max: {proving_time}')
print(f'On average one proof is required every {proof_frequency}')


Proof period length: 0:05:00
Proof validator timeout: 0:01:00
Calculating a proof takes max: 0:00:12
On average one proof is required every 1 day, 0:00:00


Storage provider assumptions:

In [10]:
number_of_slots = 1000
number_of_slotqueue_workers = 3 

print(f'Maximum number of slots: {number_of_slots}')
print(f'Provider can fill a maximum of {number_of_slotqueue_workers} slots every period')

Maximum number of slots: 1000
Provider can fill a maximum of 3 slots every period


We can now derive the proof probability:

In [11]:
proof_probability = proof_frequency // period_length

print(f'Probability that a proof is required for a slot in a period: 1 / {proof_probability}')

Probability that a proof is required for a slot in a period: 1 / 288


There is a limit to the amount of proofs that a storage provider can calculate in a single period:

In [12]:
maximum_proofs_in_period = period_length // proving_time - number_of_slotqueue_workers

print(f'Maximum number of proofs in a period: {maximum_proofs_in_period}')

Maximum number of proofs in a period: 22


We can now derive the probability that a storage provider needs to provide more proofs in a single period than it can calculate:

In [13]:
from math import comb

def probability_of_proofs_in_period(amount):
  p = 1 / proof_probability
  n = number_of_slots
  k = amount
  return comb(n, k) * pow(p, k) * pow(1-p, n-k)

def probability_of_trouble_in_period():
  trouble_range = range(maximum_proofs_in_period, number_of_slots)
  return sum(probability_of_proofs_in_period(amount) for amount in trouble_range)

def probability_of_trouble(timespan):
  number_of_periods = timespan / period_length
  return 1 - pow(1 - probability_of_trouble_in_period(), number_of_periods)

print(f'Probability of too many proofs in a single period: {probability_of_trouble(period_length)}')
print(f'Probability of too many proofs in a day: {probability_of_trouble(timedelta(days=1))}')
print(f'Probability of too many proofs in a year: {probability_of_trouble(timedelta(days=365))}')
print(f'Probability of too many proofs in a century: {probability_of_trouble(timedelta(days=100*365))}')


Probability of too many proofs in a single period: 2.1542212458314225e-11
Probability of too many proofs in a day: 6.204157187994497e-09
Probability of too many proofs in a year: 2.2645148096689383e-06
Probability of too many proofs in a century: 0.00022642609910483724


Ethereum EVM assumptions:

In [14]:
block_time = timedelta(seconds=12)

print(f'One Ethereum block every {block_time}')

One Ethereum block every 0:00:12


Marketplace contract parameters:

In [15]:
period = int(period_length.total_seconds())
downtime = int((period_length + period_timeout) / block_time) + 2

def is_prime(n):
  for i in range(2, n // 2 + 1):
    if n % i == 0:
      return False
  return True

def find_first_prime_larger_than(n):
  i = n + 1
  while not is_prime(i):
    i = i + 1
  return i

downtimeProduct = find_first_prime_larger_than(downtime)

print(f'period: {period}')
print(f'downtime: {downtime}')
print(f'downtimeProduct: {downtimeProduct}')

period: 300
downtime: 32
downtimeProduct: 37


Storage contract parameters:

In [16]:
print(f'proofProbability: {proof_probability}')

proofProbability: 288
